# Задание 1

Ноутбук `data.parquet`, сортирует данные по времени и строит числовые временные ряды пачками.

In [1]:
from pathlib import Path

import pandas as pd
import plotly.express as px

In [2]:
# Можно заменить на Path("data.parquet"), если хочешь читать parquet-файл.
file_path = Path("data.parquet")

if file_path.suffix == ".csv":
    df = pd.read_csv(file_path, parse_dates=["TS"])
elif file_path.suffix == ".parquet":
    df = pd.read_parquet(file_path, engine="pyarrow")
else:
    raise ValueError("Поддерживаются только .csv и .parquet")

df["TS"] = pd.to_datetime(df["TS"], utc=True, format="mixed")

df = df.sort_values("TS").reset_index(drop=True)

df.head()

,row,TS,Lab1_G1_N1,Lab1_G1_N2,Lab1_G1_N3,Lab1_G1_P2,Lab1_G1_T4ср,Lab1_G1_T1,Lab1_G1_T607,Lab1_G1_T600,...,Lab1_AVOM_AVOMN1,Lab1_Kp,Lab1_hGPA,Lab1_Kran_5,Lab1_Kran_2,Lab1_Kran_6,Lab1_U_Kran_GPA_A_APK,Lab1_Kran_1,Lab1_Kran_4,Lab1_q
0,1,2022-12-02 18:59:59.999998+00:00,8726,11490,5000,15.35,662.7,-16.1,36.1,67.1,...,1,0.92,0.27,0,1,0,0,1,0,50.8
1,2,2022-12-02 18:59:59.999998+00:00,8726,11490,5000,15.35,662.7,-16.1,36.1,67.1,...,1,0.92,0.27,0,1,0,0,1,0,50.8
2,3,2022-12-02 19:00:59.999998+00:00,8726,11489,4993,15.35,662.4,-16.4,36.1,67.0,...,1,0.92,0.27,0,1,0,0,1,0,50.8
3,4,2022-12-02 19:00:59.999998+00:00,8726,11489,4993,15.35,662.4,-16.4,36.1,67.0,...,1,0.92,0.27,0,1,0,0,1,0,50.8
4,5,2022-12-02 19:01:59.999999+00:00,8725,11493,5001,15.35,663.1,-16.0,36.1,67.0,...,1,0.92,0.27,0,1,0,0,1,0,50.8


In [3]:
numeric_cols = df.select_dtypes(include="number").columns.tolist()

# row - служебная колонка, ее не рассматриваем как временной ряд.
numeric_cols = [col for col in numeric_cols if col != "row"]

print(f"Найдено числовых временных рядов: {len(numeric_cols)}")

Найдено числовых временных рядов: 117


## Есть ли пропуски?

In [4]:
# Проверка пропусков и константных признаков

missing = df.isna().sum()
missing_nonzero = missing[missing > 0]

nunique = df.nunique(dropna=False)
constant_cols = nunique[nunique == 1].index.tolist()

print(f"Размер данных: {df.shape[0]} строк, {df.shape[1]} колонок")
print(f"Всего формальных пропусков NaN/None/NaT: {missing.sum()}")
print(f"Колонок с пропусками: {len(missing_nonzero)}")
print(f"Константных колонок: {len(constant_cols)}")

constant_report = pd.DataFrame({
    "column": constant_cols,
    "constant_value": [df[col].iloc[0] for col in constant_cols],
})

display(constant_report)

Размер данных: 2878 строк, 119 колонок
Всего формальных пропусков NaN/None/NaT: 0
Колонок с пропусками: 0
Константных колонок: 33


,column,constant_value
0,Lab1_G2_Fнш,0.00
1,Lab1_G2_Fма,0.00
2,Lab1_G2_Fмн,0.00
3,Lab1_G2_Fств,0.00
4,Lab1_G3_Маслосистема,0.00
5,Lab1_Lmp_Txt_AO,0.00
6,Lab1_Lmp_Txt_BEAO,0.00
7,Lab1_TC_dPvfKVOU,0.60
8,Lab1_Lmp_Txt_BK1,0.00
9,Lab1_Lmp_Txt_BK2,0.00


## Есть ли дубли?

In [5]:
# Проверка полных дублей строк
n_full_duplicates = df.drop(columns=["row"]).duplicated().sum()

print(f"Полных дублей строк: {n_full_duplicates}")

# Проверка дублей по временной метке

ts_duplicates_count = df.duplicated(subset=["TS"]).sum()

print(f"Дублей по TS: {ts_duplicates_count}")

df[df.duplicated(subset=["TS"], keep=False)].sort_values("TS").head()

if n_full_duplicates == 0:
    print("Полные дубли строк отсутствуют.")
else:
    print("Обнаружены полные дубли строк, требуется дополнительная проверка.")
    duplicates = df[df.drop(columns=["row"]).duplicated(keep=False)]
    print(duplicates)
    

Полных дублей строк: 1439
Дублей по TS: 1439
Обнаружены полные дубли строк, требуется дополнительная проверка.
       row                               TS  Lab1_G1_N1  Lab1_G1_N2  \
0        1 2022-12-02 18:59:59.999998+00:00        8726       11490   
1        2 2022-12-02 18:59:59.999998+00:00        8726       11490   
2        3 2022-12-02 19:00:59.999998+00:00        8726       11489   
3        4 2022-12-02 19:00:59.999998+00:00        8726       11489   
4        5 2022-12-02 19:01:59.999999+00:00        8725       11493   
...    ...                              ...         ...         ...   
2873  2874        2022-12-03 18:56:00+00:00        8686       11447   
2874  2875 2022-12-03 18:56:59.999999+00:00        8690       11449   
2875  2876 2022-12-03 18:56:59.999999+00:00        8690       11449   
2876  2877        2022-12-03 18:58:00+00:00        8690       11449   
2877  2878        2022-12-03 18:58:00+00:00        8690       11449   

      Lab1_G1_N3  Lab1_G1_P2  Lab1_G

## Удаляем полные дубли

In [6]:
df = (
    df
    .drop(columns=["row"])
    .drop_duplicates()
    .reset_index(drop=True)
)

# Факторизация

Задаём множество временных рядов
один столбец = один временной ряд

In [14]:
# Множество временных рядов X:
# каждый числовой столбец, кроме служебных, считаем отдельным временным рядом.

service_cols = ["TS", "row"] 

numeric_cols = df.select_dtypes(include="number").columns.tolist()

series_cols = [
    col for col in numeric_cols
    if col not in service_cols
]

print("Количество временных рядов:", len(series_cols))

Количество временных рядов: 117


Шаг 2. Делим ряды на константные и неконстантные

In [11]:
constant_cols = []
nonconstant_cols = []

for col in series_cols:
    n_unique = df[col].nunique(dropna=True)

    if n_unique == 1:
        constant_cols.append(col)
    else:
        nonconstant_cols.append(col)

print("Константных рядов:", len(constant_cols))
print("Неконстантных рядов:", len(nonconstant_cols))

Константных рядов: 33
Неконстантных рядов: 84


### Константные ряды

Шаг 3. Группировка константных рядв по значению

xi ~ xj, если xi(t) = xj(t) для всех t

In [12]:
constant_classes = {}

for col in constant_cols:
    value = df[col].dropna().iloc[0]

    if value not in constant_classes:
        constant_classes[value] = []

    constant_classes[value].append(col)

print("Классы эквивалентности для константных рядов:")

for value, members in constant_classes.items():
    print(f"Значение {value}: {len(members)} рядов")
    print(members)
    print()

Классы эквивалентности для константных рядов:
Значение 0: 17 рядов
['Lab1_G2_Fнш', 'Lab1_G2_Fма', 'Lab1_G2_Fмн', 'Lab1_G2_Fств', 'Lab1_G3_Маслосистема', 'Lab1_Lmp_Txt_AO', 'Lab1_Lmp_Txt_BEAO', 'Lab1_Lmp_Txt_BK1', 'Lab1_Lmp_Txt_BK2', 'Lab1_Kran_11', 'Lab1_Kran_PZS', 'Lab1_VOD2', 'Lab1_Kran_9', 'Lab1_Kran_5', 'Lab1_Kran_6', 'Lab1_U_Kran_GPA_A_APK', 'Lab1_Kran_4']

Значение 0.6: 1 рядов
['Lab1_TC_dPvfKVOU']

Значение 1: 11 рядов
['Lab1_TC_VKPGV', 'Lab1_TC_VPOS', 'Lab1_Kran_10', 'Lab1_VOD1', 'Lab1_Kran_12', 'Lab1_U_Kran_GPA_A_OGK', 'Lab1_Kran_SK', 'Lab1_AVOM_AVOMD1', 'Lab1_AVOM_AVOMN1', 'Lab1_Kran_2', 'Lab1_Kran_1']

Значение 26: 1 рядов
['Lab1_Tpg']

Значение 0.92: 1 рядов
['Lab1_Kp']

Значение 0.27: 1 рядов
['Lab1_hGPA']

Значение 50.8: 1 рядов
['Lab1_q']



Шаг 4. Выбор представителя из каждого класса константных рядов

In [13]:
constant_representatives = []

constant_report_rows = []

for value, members in constant_classes.items():
    representative = members[0]

    constant_representatives.append(representative)

    constant_report_rows.append({
        "type": "constant",
        "constant_value": value,
        "representative": representative,
        "class_size": len(members),
        "members": members
    })

constant_report = pd.DataFrame(constant_report_rows)

display(constant_report)

print("Количество классов константных рядов:", len(constant_report))
print("Количество представителей:", len(constant_representatives))

,type,constant_value,representative,class_size,members
0,constant,0.00,Lab1_G2_Fнш,17,"[Lab1_G2_Fнш, Lab1_G2_Fма, Lab1_G2_Fмн, Lab1_G..."
1,constant,0.60,Lab1_TC_dPvfKVOU,1,[Lab1_TC_dPvfKVOU]
2,constant,1.00,Lab1_TC_VKPGV,11,"[Lab1_TC_VKPGV, Lab1_TC_VPOS, Lab1_Kran_10, La..."
3,constant,26.00,Lab1_Tpg,1,[Lab1_Tpg]
4,constant,0.92,Lab1_Kp,1,[Lab1_Kp]
5,constant,0.27,Lab1_hGPA,1,[Lab1_hGPA]
6,constant,50.80,Lab1_q,1,[Lab1_q]


Количество классов константных рядов: 7
Количество представителей: 7


### Неконстантные ряды

In [16]:
import numpy as np
import pandas as pd

In [17]:
# Проверяем стационарность неконстантных рядов.
# Стационарный ряд: среднее и разброс примерно стабильны во времени.

N_WINDOWS = 6

MEAN_THRESHOLD = 0.15
STD_THRESHOLD = 0.35
TREND_THRESHOLD = 0.20


def stationarity_by_windows(values):
    values = pd.Series(values).dropna().to_numpy(dtype=float)

    n = len(values)

    amplitude = values.max() - values.min()
    global_std = values.std()

    # если ряд почти константный, считаем его стационарным
    if amplitude == 0 or global_std == 0:
        return True, 0, 0, 0

    windows = np.array_split(values, N_WINDOWS)

    window_means = np.array([w.mean() for w in windows])
    window_stds = np.array([w.std() for w in windows])

    # насколько меняется среднее по окнам
    mean_change = window_means.std() / amplitude

    # насколько меняется разброс по окнам
    std_change = window_stds.std() / global_std

    # насколько выражен общий линейный тренд
    t = np.arange(n)
    slope = np.polyfit(t, values, 1)[0]
    trend_strength = abs(slope) * n / amplitude

    is_stationary = (
        mean_change <= MEAN_THRESHOLD
        and std_change <= STD_THRESHOLD
        and trend_strength <= TREND_THRESHOLD
    )

    return is_stationary, mean_change, std_change, trend_strength


stationary_cols = []
nonstationary_cols = []

stationarity_rows = []

for col in nonconstant_cols:
    is_stationary, mean_change, std_change, trend_strength = stationarity_by_windows(df[col])

    if is_stationary:
        stationary_cols.append(col)
    else:
        nonstationary_cols.append(col)

    stationarity_rows.append({
        "series_name": col,
        "is_stationary": is_stationary,
        "mean_change": mean_change,
        "std_change": std_change,
        "trend_strength": trend_strength
    })


stationarity_report = pd.DataFrame(stationarity_rows)

print("Стационарных неконстантных рядов:", len(stationary_cols))
print("Нестационарных неконстантных рядов:", len(nonstationary_cols))

display(
    stationarity_report
    .sort_values("is_stationary", ascending=False)
    .head(20)
)

Стационарных неконстантных рядов: 40
Нестационарных неконстантных рядов: 44


,series_name,is_stationary,mean_change,std_change,trend_strength
42,Lab1_G3_V1,True,0.039712,0.199285,0.053460
25,Lab1_G2_3_77F2,True,0.018666,0.080512,0.029289
27,Lab1_G2_Fc9,True,0.042026,0.055570,0.114839
28,Lab1_G2_Fс8,True,0.037219,0.069543,0.077517
30,Lab1_G2_2F3,True,0.071483,0.094687,0.019347
31,Lab1_G2_3F3,True,0.050544,0.092486,0.066903
32,Lab1_G2_Fтк9,True,0.011456,0.063410,0.017339
33,Lab1_G2_Fтк8,True,0.017403,0.043821,0.005448
34,Lab1_G2_Fн9,True,0.024023,0.112116,0.020756
36,Lab1_G2_VoСТ,True,0.049322,0.051828,0.135157


In [21]:
!pip install statsmodels

   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.5 MB ? eta -:--:--
   ---- ----------------------------------- 1.0/9.5 MB 2.6 MB/s eta 0:00:04
   ------ --------------------------------- 1.6/9.5 MB 2.8 MB/s eta 0:00:03
   --------- ------------------------------ 2.4/9.5 MB 3.0 MB/s eta 0:00:03
   ------------ --------------------------- 2.9/9.5 MB 3.0 MB/s eta 0:00:03
   ---------------- ----------------------- 3.9/9.5 MB 3.2 MB/s eta 0:00:02
   ------------------- -------------------- 4.7/9.5 MB 3.3 MB/s eta 0:00:02
   ------------------------ --------------- 5.8/9.5 MB 3.5 MB/s eta 0:00:02
   ---------------------------- ----------- 6.8/9.5 MB 3.7 MB/s eta 0:00:01
   ---------------------------------- ----- 8.1/9.5 MB 3.9 MB/s eta 0:00:01
   -------------------------------------- - 9.2/9.5 MB 4.1 MB/s eta 0:00:01
   ------------------------------

In [18]:
# Проверяем, как меняется количество стационарных рядов
# при разных порогах.

threshold_sets = [
    {
        "name": "strict",
        "mean_threshold": 0.10,
        "std_threshold": 0.25,
        "trend_threshold": 0.15,
    },
    {
        "name": "base",
        "mean_threshold": 0.15,
        "std_threshold": 0.35,
        "trend_threshold": 0.20,
    },
    {
        "name": "soft",
        "mean_threshold": 0.20,
        "std_threshold": 0.45,
        "trend_threshold": 0.30,
    },
]

sensitivity_rows = []

for params in threshold_sets:
    stationary_count = 0
    nonstationary_count = 0

    for _, row in stationarity_report.iterrows():
        is_stationary_current = (
            row["mean_change"] <= params["mean_threshold"]
            and row["std_change"] <= params["std_threshold"]
            and row["trend_strength"] <= params["trend_threshold"]
        )

        if is_stationary_current:
            stationary_count += 1
        else:
            nonstationary_count += 1

    sensitivity_rows.append({
        "variant": params["name"],
        "mean_threshold": params["mean_threshold"],
        "std_threshold": params["std_threshold"],
        "trend_threshold": params["trend_threshold"],
        "stationary_count": stationary_count,
        "nonstationary_count": nonstationary_count,
    })

sensitivity_report = pd.DataFrame(sensitivity_rows)

display(sensitivity_report)

,variant,mean_threshold,std_threshold,trend_threshold,stationary_count,nonstationary_count
0,strict,0.10,0.25,0.15,28,56
1,base,0.15,0.35,0.20,40,44
2,soft,0.20,0.45,0.30,64,20


In [22]:
# Проверяем стационарность неконстантных рядов через ADF и KPSS.
#
# ADF:
# H0: ряд имеет unit root, то есть ряд нестационарен.
# Если p-value < 0.05, отвергаем H0 и считаем ряд стационарным.
#
# KPSS:
# H0: ряд стационарен.
# Если p-value >= 0.05, не отвергаем H0 и считаем ряд стационарным.
#
# Поэтому устойчиво стационарный ряд:
# ADF p-value < 0.05 И KPSS p-value >= 0.05


from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tools.sm_exceptions import InterpolationWarning

import warnings


ALPHA = 0.05

stationarity_rows = []

for col in nonconstant_cols:
    values = pd.Series(df[col]).dropna().to_numpy(dtype=float)

    # ADF-тест
    try:
        adf_result = adfuller(values, autolag="AIC")
        adf_pvalue = adf_result[1]
        adf_stationary = adf_pvalue < ALPHA
    except Exception:
        adf_pvalue = np.nan
        adf_stationary = False

    # KPSS-тест
    # regression="c" — проверяем стационарность вокруг постоянного уровня.
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", InterpolationWarning)
            kpss_result = kpss(values, regression="c", nlags="auto")

        kpss_pvalue = kpss_result[1]
        kpss_stationary = kpss_pvalue >= ALPHA
    except Exception:
        kpss_pvalue = np.nan
        kpss_stationary = False

    # Итоговая классификация
    if adf_stationary and kpss_stationary:
        stationarity_type = "stationary"

    elif (not adf_stationary) and (not kpss_stationary):
        stationarity_type = "nonstationary"

    elif (not adf_stationary) and kpss_stationary:
        stationarity_type = "trend_stationary_or_borderline"

    elif adf_stationary and (not kpss_stationary):
        stationarity_type = "difference_stationary_or_borderline"

    else:
        stationarity_type = "undefined"

    stationarity_rows.append({
        "series_name": col,
        "adf_pvalue": adf_pvalue,
        "adf_stationary": adf_stationary,
        "kpss_pvalue": kpss_pvalue,
        "kpss_stationary": kpss_stationary,
        "stationarity_type": stationarity_type
    })


stationarity_report = pd.DataFrame(stationarity_rows)

display(stationarity_report)

print("Количество рядов по типам стационарности:")
display(
    stationarity_report["stationarity_type"]
    .value_counts()
    .to_frame("count")
)

,series_name,adf_pvalue,adf_stationary,kpss_pvalue,kpss_stationary,stationarity_type
0,Lab1_G1_N1,5.097505e-03,True,0.010000,False,difference_stationary_or_borderline
1,Lab1_G1_N2,3.097609e-02,True,0.011368,False,difference_stationary_or_borderline
2,Lab1_G1_N3,1.416737e-25,True,0.100000,True,stationary
3,Lab1_G1_P2,1.412013e-02,True,0.010000,False,difference_stationary_or_borderline
4,Lab1_G1_T4ср,4.209119e-02,True,0.021574,False,difference_stationary_or_borderline
...,...,...,...,...,...,...
79,Lab1_TposleNag,9.903337e-01,False,0.010000,False,nonstationary
80,Lab1_PdoNag,8.585527e-01,False,0.010000,False,nonstationary
81,Lab1_TdoNag,9.446488e-01,False,0.010000,False,nonstationary
82,Lab1_Rc,9.621477e-01,False,0.010000,False,nonstationary


Количество рядов по типам стационарности:


,count
stationarity_type,
difference_stationary_or_borderline,36
stationary,23
nonstationary,23
trend_stationary_or_borderline,2


In [24]:
import numpy as np
import pandas as pd

# Количество окон, на которые делим временной ряд
N_WINDOWS = 6


def minmax_normalize(values):
    values = pd.Series(values).dropna().to_numpy(dtype=float)

    min_value = values.min()
    max_value = values.max()

    amplitude = max_value - min_value

    if amplitude == 0:
        return None

    return (values - min_value) / amplitude


def stationarity_by_eps(values, eps_mean=0.05, eps_std=0.05, eps_trend=0.05):
    # Нормируем ряд к диапазону [0, 1]
    y = minmax_normalize(values)

    # Если ряд константный, он стационарный
    if y is None:
        return True, 0, 0, 0

    n = len(y)

    # Делим ряд на окна
    windows = np.array_split(y, N_WINDOWS)

    # Считаем среднее и стандартное отклонение в каждом окне
    window_means = np.array([w.mean() for w in windows])
    window_stds = np.array([w.std() for w in windows])

    # Изменение среднего по окнам
    mean_change = window_means.max() - window_means.min()

    # Изменение стандартного отклонения по окнам
    std_change = window_stds.max() - window_stds.min()

    # Сила линейного тренда на нормированном ряде
    t = np.arange(n)
    slope = np.polyfit(t, y, 1)[0]
    trend_strength = abs(slope) * (n - 1)

    is_stationary = (
        mean_change <= eps_mean
        and std_change <= eps_std
        and trend_strength <= eps_trend
    )

    return is_stationary, mean_change, std_change, trend_strength

In [29]:
eps_values = [0.001, 0.01, 0.05, 0.10]

eps_rows = []
stationary_by_eps = {}

for eps in eps_values:
    stationary_count = 0
    nonstationary_count = 0

    current_stationary = []
    current_nonstationary = []

    for col in nonconstant_cols:
        is_stationary, mean_change, std_change, trend_strength = stationarity_by_eps(
            df[col],
            eps_mean=eps,
            eps_std=eps,
            eps_trend=eps
        )

        if is_stationary:
            stationary_count += 1
            current_stationary.append(col)
        else:
            nonstationary_count += 1
            current_nonstationary.append(col)

    eps_rows.append({
        "eps": eps,
        "stationary_count": stationary_count,
        "nonstationary_count": nonstationary_count,
        "stationary_series": current_stationary
    })

    stationary_by_eps[eps] = current_stationary


eps_report = pd.DataFrame(eps_rows)

display(eps_report)

,eps,stationary_count,nonstationary_count,stationary_series
0,0.001,0,84,[]
1,0.010,0,84,[]
2,0.050,7,77,"[Lab1_G1_N3, Lab1_G2_Fтк2, Lab1_G2_Fтк9, Lab1_..."
3,0.100,16,68,"[Lab1_G1_N3, Lab1_G2_Fc2, Lab1_G2_Fтк2, Lab1_G..."


In [30]:
for eps, cols in stationary_by_eps.items():
    print("=" * 80)
    print(f"eps = {eps}")
    print("Количество стационарных рядов:", len(cols))
    print(cols)

eps = 0.001
Количество стационарных рядов: 0
[]
eps = 0.01
Количество стационарных рядов: 0
[]
eps = 0.05
Количество стационарных рядов: 7
['Lab1_G1_N3', 'Lab1_G2_Fтк2', 'Lab1_G2_Fтк9', 'Lab1_G3_N3', 'Lab1_G3_dPf1', 'Lab1_G3_Pc1', 'Lab1_TC_P615']
eps = 0.1
Количество стационарных рядов: 16
['Lab1_G1_N3', 'Lab1_G2_Fc2', 'Lab1_G2_Fтк2', 'Lab1_G2_Fцс', 'Lab1_G2_Fкпа', 'Lab1_G2_Fтк4', 'Lab1_G2_3_77F2', 'Lab1_G2_Fтк9', 'Lab1_G2_Fтк8', 'Lab1_G2_Fн9', 'Lab1_G3_N3', 'Lab1_G3_dPf1', 'Lab1_G3_Pm', 'Lab1_G3_V2', 'Lab1_G3_Pc1', 'Lab1_TC_P615']


### Проверяем что дублей нет...

In [7]:
# Проверка полных дублей строк
n_full_duplicates = df.duplicated().sum()

print(f"Полных дублей строк: {n_full_duplicates}")

# Проверка дублей по временной метке

ts_duplicates_count = df.duplicated(subset=["TS"]).sum()

print(f"Дублей по TS: {ts_duplicates_count}")

df[df.duplicated(subset=["TS"], keep=False)].sort_values("TS").head()

if n_full_duplicates == 0:
    print("Полные дубли строк отсутствуют.")
else:
    print("Обнаружены полные дубли строк, требуется дополнительная проверка.")
    duplicates = df[df.drop(columns=["row"]).duplicated(keep=False)]
    print(duplicates)
    

Полных дублей строк: 0
Дублей по TS: 0
Полные дубли строк отсутствуют.


## Проверка временного шага

In [8]:
time_deltas = df["TS"].diff().dropna()

print("Частоты временных интервалов между соседними наблюдениями:")
display(time_deltas.value_counts().sort_index())

Частоты временных интервалов между соседними наблюдениями:


TS
0 days 00:00:59.999998      5
0 days 00:00:59.999999    372
0 days 00:01:00           691
0 days 00:01:00.000001    356
0 days 00:01:00.000002     14
Name: count, dtype: int64

In [9]:
rounded_time_deltas = time_deltas.dt.total_seconds().round()

print("Частоты временных интервалов после округления до секунд:")
display(rounded_time_deltas.value_counts().sort_index())

Частоты временных интервалов после округления до секунд:


TS
60.0    1438
Name: count, dtype: int64

Интервалы между соседними наблюдениями близки к одной минуте. Небольшие отличия на уровне микросекунд связаны с технической точностью записи временных меток. После округления до секунд все интервалы соответствуют 60 секундам, поэтому временной ряд можно рассматривать как равномерно дискретизированный с шагом около одной минуты.

# Визуализация

In [8]:
print(f"Найдем колонки где данные принимают только 1 значение.")
# Количество уникальных значений в каждой колонке
nunique = df.nunique(dropna=False)

# Колонки, где только одно уникальное значение
constant_cols = nunique[nunique == 1].index.tolist()

print(f"Константных колонок: {len(constant_cols)}")
constant_cols

df_for_visualization = df.drop(columns=constant_cols)

print(f"Было колонок: {df.shape[1]}")
print(f"Стало колонок: {df_for_visualization.shape[1]}")
print(f"Удалено константных колонок: {len(constant_cols)}")

visualization_cols = df_for_visualization.columns

batch_size = 10

for i in range(0, len(visualization_cols), batch_size):
    cols_batch = visualization_cols[i:i + batch_size]

    fig = px.line(
        df_for_visualization,
        x="TS",
        y=cols_batch,
        title=f"Временные ряды {i + 1}-{i + len(cols_batch)}",
    )

    fig.update_layout(
        height=600,
        xaxis_title="Время",
        yaxis_title="Значение",
        legend_title="Показатель",
    )

    fig.show()
    
constant_values = (
    df[constant_cols]
    .iloc[0]
    .reset_index()
)

constant_values.columns = ["column", "constant_value"]

constant_values_sorted = constant_values.sort_values(
    "constant_value",
    ascending=True
)

fig = px.bar(
    constant_values_sorted,
    x="column",
    y="constant_value",
    title="Константные признаки и их значения",
)

fig.update_layout(
    height=500,
    xaxis_title="Признак",
    yaxis_title="Постоянное значение",
    xaxis_tickangle=-45,
)

fig.show()

Найдем колонки где данные принимают только 1 значение.
Константных колонок: 33
Было колонок: 118
Стало колонок: 85
Удалено константных колонок: 33


## Сохраним dataset после предобработки

In [10]:
df.to_parquet("dataset_clean.parquet", index=False)
df.to_csv("dataset_clean.csv", index=False, encoding="utf-8-sig")